Libraries

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sb
import requests
import time
import os
import dask.dataframe as dd
from IPython.display import display

pd.set_option('display.max_columns', None)

In [27]:
# Definir rango de fechas para Enero 2021

start_date = '2021-01-01'
end_date = '2021-01-31'
# end_date = '2022-12-31'

date_range = pd.date_range(start=start_date, end=end_date)
# print(date_range)

In [28]:
# Importar Enero 2021 - forma 1

# Definir lista de DF's y tiempo inicial de carga
start_time = time.time()
df = []

for single_date in date_range:
    url = f'https://raw.githubusercontent.com/CSSEGISandData/COVID-19/refs/heads/master/csse_covid_19_data/csse_covid_19_daily_reports/{single_date.strftime("%m-%d-%Y")}.csv'
    df.append(pd.read_csv(url))

# Combinar los DataFrames diarios en uno solo.
enero = pd.concat(df, ignore_index=True)

# Calcular tiempo de carga
load_time_1 = time.time() - start_time
print(f"Tiempo de carga: {load_time_1:.2f} s")

Tiempo de carga: 1.90 s


In [29]:
## Instalar aiohttp (necesario para fsspec HTTPFileSystem usado por Dask/pandas)
#%pip install aiohttp -q
#
#import aiohttp

In [30]:
# Importar Enero 2021 - forma 2 (con Dask)

# Definir lista de DF's y tiempo inicial de carga
start_time = time.time()
df2 = []

for single_date in date_range:
    url = f'https://raw.githubusercontent.com/CSSEGISandData/COVID-19/refs/heads/master/csse_covid_19_data/csse_covid_19_daily_reports/{single_date.strftime("%m-%d-%Y")}.csv'
    df2.append(dd.read_csv(url, dtype={'Admin2': 'object'}))

# Combinar los DataFrames diarios en uno solo.
enero = dd.concat(df2, ignore_index=True)

# Calcular tiempo de carga
load_time_2 = time.time() - start_time
print(f"Tiempo de carga con Dask: {load_time_2:.2f} s")

Tiempo de carga con Dask: 1.32 s


In [31]:
# 1. Cargar y visualizar los primeros 5 registros

enero.head()

,FIPS,Admin2,Province_State,Country_Region,Last_Update,Lat,Long_,Confirmed,Deaths,Recovered,Active,Combined_Key,Incident_Rate,Case_Fatality_Ratio
0,NaN,<NA>,<NA>,Afghanistan,2021-01-02 05:22:33,33.93911,67.709953,52513,2201,41727,8585,Afghanistan,134.896578,4.191343
1,NaN,<NA>,<NA>,Albania,2021-01-02 05:22:33,41.15330,20.168300,58316,1181,33634,23501,Albania,2026.409062,2.025173
2,NaN,<NA>,<NA>,Algeria,2021-01-02 05:22:33,28.03390,1.659600,99897,2762,67395,29740,Algeria,227.809861,2.764848
3,NaN,<NA>,<NA>,Andorra,2021-01-02 05:22:33,42.50630,1.521800,8117,84,7463,570,Andorra,10505.403482,1.034865
4,NaN,<NA>,<NA>,Angola,2021-01-02 05:22:33,-11.20270,17.873900,17568,405,11146,6017,Angola,53.452981,2.305328


In [32]:
# 2. Mostrar el número total de filas y columnas del DataFrame.

print('Filas en total: ', len(enero))
print('Columnas en total: ', len(enero.columns))

Filas en total:  124398
Columnas en total:  14


In [33]:
# 3. Describir los tipos de datos (dtypes) y convertir las columnas necesarias (por ejemplo,
# fechas).

enero.dtypes

FIPS                           float64
Admin2                 string[pyarrow]
Province_State         string[pyarrow]
Country_Region         string[pyarrow]
Last_Update            string[pyarrow]
Lat                            float64
Long_                          float64
Confirmed                        int64
Deaths                           int64
Recovered                        int64
Active                           int64
Combined_Key           string[pyarrow]
Incident_Rate                  float64
Case_Fatality_Ratio            float64
dtype: object

In [34]:
# 3

# Formatear la columna Last_Update a tipo datetime

enero = enero.assign(Last_Update=dd.to_datetime(enero['Last_Update'], errors='coerce'))

enero.dtypes

FIPS                           float64
Admin2                 string[pyarrow]
Province_State         string[pyarrow]
Country_Region         string[pyarrow]
Last_Update             datetime64[ns]
Lat                            float64
Long_                          float64
Confirmed                      float64
Deaths                         float64
Recovered                      float64
Active                         float64
Combined_Key           string[pyarrow]
Incident_Rate                  float64
Case_Fatality_Ratio            float64
dtype: object

In [35]:
# 3

# Uso de memoria antes de conversión de tipos

enero_pdf = enero.compute()

# Verificar memoria usada por el DataFrame pandas resultante
memoria = enero_pdf.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"Memoria usada del DataFrame: {memoria:.2f} MB")

Memoria usada del DataFrame: 18.69 MB


In [36]:
# 3

enero['Province_State'] = enero['Province_State'].astype('category')
enero['Country_Region'] = enero['Country_Region'].astype('category')
enero['Combined_Key'] = enero['Combined_Key'].astype('category')

enero_pdf = enero.compute()

# Verificar memoria usada por el DataFrame pandas resultante
memoria_optimizacion = enero_pdf.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"Memoria usada del DataFrame tras optimización: {memoria_optimizacion:.2f} MB")

Memoria usada del DataFrame tras optimización: 13.08 MB


In [37]:
print(f"Diferencia en uso de memoria: {(memoria - memoria_optimizacion):.2f} MB")

Diferencia en uso de memoria: 5.61 MB


In [38]:
# 4. Detectar y mostrar valores nulos o faltantes por columna.

display(enero.isnull().sum())

Dask Series Structure:
npartitions=1
Active       int64
Recovered      ...
Dask Name: sum, 78 expressions
Expr=(~ NotNull(frame=Assign(frame=Assign(frame=Assign(frame=Assign(frame=Concat(frames=[ArrowStringConversion(frame=FromMapProjectable(e614fcf)), ArrowStringConversion(frame=FromMapProjectable(97a2c93)), ArrowStringConversion(frame=FromMapProjectable(cccfefa)), ArrowStringConversion(frame=FromMapProjectable(41ca111)), ArrowStringConversion(frame=FromMapProjectable(d746a90)), ArrowStringConversion(frame=FromMapProjectable(e74e584)), ArrowStringConversion(frame=FromMapProjectable(ecf6a94)), ArrowStringConversion(frame=FromMapProjectable(5f903b0)), ArrowStringConversion(frame=FromMapProjectable(1722485)), ArrowStringConversion(frame=FromMapProjectable(3391df3)), ArrowStringConversion(frame=FromMapProjectable(6e939b2)), ArrowStringConversion(frame=FromMapProjectable(44e5435)), ArrowStringConversion(frame=FromMapProjectable(d30d18a)), ArrowStringConversion(frame=FromMapProjectable(9edf

In [39]:
# 5. Eliminar columnas irrelevantes (por ejemplo, códigos FIPS o coordenadas si no se usarán).

enero = enero.drop(columns=['FIPS', 'Admin2', 'Lat', 'Long_', 'Combined_Key'])
enero.head(1)

,Province_State,Country_Region,Last_Update,Confirmed,Deaths,Recovered,Active,Incident_Rate,Case_Fatality_Ratio
0,<NA>,Afghanistan,2021-01-02 05:22:33,52513,2201,41727,8585,134.896578,4.191343


In [40]:
# 6. Estandarizar nombres de columnas (usar formato snake_case).

enero.columns = enero.columns.str.lower().str.replace(' ', '_')
enero.head(1)

,province_state,country_region,last_update,confirmed,deaths,recovered,active,incident_rate,case_fatality_ratio
0,<NA>,Afghanistan,2021-01-02 05:22:33,52513,2201,41727,8585,134.896578,4.191343


In [41]:
# 7. Homogeneizar nombres de países (ej. “US” → “United States”).

enero['country_region'] = enero['country_region'].replace({'US': 'United States'})
enero[enero['country_region'] == 'United States'].head(1)

,province_state,country_region,last_update,confirmed,deaths,recovered,active,incident_rate,case_fatality_ratio
648,Alabama,United States,2021-01-02 05:22:33,4239,50,0,4189,7587.391935,1.179523


In [42]:
# 8. Convertir la columna last_update al formato YYYY-MM-DD (día preciso)
# Asegurar datetime y mantener sólo fecha (YYYY-MM-DD)
enero['last_update'] = dd.to_datetime(enero['last_update'], errors='coerce').dt.date
enero.head(1)

,province_state,country_region,last_update,confirmed,deaths,recovered,active,incident_rate,case_fatality_ratio
0,<NA>,Afghanistan,2021-01-02,52513,2201,41727,8585,134.896578,4.191343


In [43]:
# 9. Crear una columna active_cases = Confirmed - Deaths - Recovered.

enero['active_cases'] = (enero['confirmed'] - enero['deaths'] - enero['recovered'])
enero.head(1)

,province_state,country_region,last_update,confirmed,deaths,recovered,active,incident_rate,case_fatality_ratio,active_cases
0,<NA>,Afghanistan,2021-01-02,52513,2201,41727,8585,134.896578,4.191343,8585


In [45]:
# 10. Guardar el DataFrame limpio como covid_clean_enero2020.csv e indicar su tamaño en MB.

try:
    enero.compute().to_csv('covid_clean_enero2020.csv')
except:
    os.remove('covid_clean_enero2020.csv')
    enero.compute().to_csv('covid_clean_enero2020.csv')

file_size = os.path.getsize('covid_clean_enero2020.csv') / (1024 * 1024)  # Convertir a MB
print(f'El tamaño del archivo covid_clean_enero2020.csv es: {file_size:.2f} MB')

El tamaño del archivo covid_clean_enero2020.csv es: 11.03 MB
